# Word Order Experiments

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import os
import re

import dotenv
import evaluate
import numpy as np
import pandas as pd
import pyrootutils
import tiktoken
from transformers import AutoTokenizer

PROJECT_ROOT = path = pyrootutils.find_root(indicator=".project-root")
DATA_DIR = PROJECT_ROOT / "data"
BATCHES_DIR = PROJECT_ROOT / "batches"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
FIGURES_DIR = PROJECT_ROOT / "paper" / "figures"
EXP_DIR = BATCHES_DIR / "size_exp"

# create figures directory if it doesn't exist
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

dotenv.load_dotenv(PROJECT_ROOT / ".env")
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if HF_TOKEN and not os.getenv("HUGGINGFACE_HUB_TOKEN"):
    os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

## Define aesthetics

In [ ]:
import aesthetics as aes  # noqa: F401

sns = aes.sns
mtick = aes.mtick
plt = aes.plt

aes.PALETTE_METRICS
aes.PALETTE_MODELS

## Grammars & Samples

In [ ]:
EXPERIMENT_GRAMMARS = DATA_DIR / "size_exp" / "size_grammars.txt"

grammar_ids = []
with open(EXPERIMENT_GRAMMARS, "r") as f:
    grammar_ids = [line.strip() for line in f.readlines()]


grammar_files = [f"grammar_{gid}.json" for gid in grammar_ids]
sample_files = [f"samples_{gid}.jsonl" for gid in grammar_ids]

grammars = []
for gf in grammar_files:
    with open(DATA_DIR / "size_exp" / gf, "r") as f:
        grammars.append(json.load(f))

samples = []
for path in sample_files:
    with open(DATA_DIR / "size_exp" / path, "r") as f:
        samples.extend([json.loads(line) for line in f])
samples_df = pd.DataFrame(samples)
samples_df["input_length"] = samples_df["left_phonetic"].apply(lambda x: len(x.split()))
samples_df["clauses"] = samples_df["depth"] + 1

grammar_sizes = pd.DataFrame(
    [{"name": g["name"], "n_words": g["n_words"]} for g in grammars]
)

## Outputs

In [ ]:
tokenizers = {
    "gpt-5-nano": tiktoken.encoding_for_model("gpt-5-nano"),
    "gpt-5-mini": tiktoken.encoding_for_model("gpt-5-mini"),
    "gpt-5": tiktoken.encoding_for_model("gpt-5"),
    "google/gemma-3-1b-it": AutoTokenizer.from_pretrained(
        "google/gemma-3-1b-it", token=HF_TOKEN
    ),
    "google/gemma-3-4b-it": AutoTokenizer.from_pretrained(
        "google/gemma-3-4b-it", token=HF_TOKEN
    ),
    "google/gemma-3-12b-it": AutoTokenizer.from_pretrained(
        "google/gemma-3-12b-it", token=HF_TOKEN
    ),
}


def fuzzy_model(model: str | None) -> str:
    return re.sub(r"-\d{4}-\d{2}-\d{2}$", "", model or "")


def metadata_key(custom_id: str | None) -> str | None:
    if not custom_id:
        return None
    match = re.match(
        r"^(?P<grammar>[0-9a-f]+)-[0-9a-f]+-(?:request|sample)-(?P<sample_id>\d+)$",
        custom_id,
    )
    if match is None:
        return custom_id
    return f"{match.group('grammar')}-sample-{match.group('sample_id')}"

In [ ]:
outputs_glob = sorted(EXP_DIR.glob("*_output.jsonl"))
inputs_glob = sorted(
    path
    for path in EXP_DIR.glob("inputs_*.jsonl")
    if not path.name.endswith("_output.jsonl")
)

output_frames = []
for path in outputs_glob:
    df = pd.read_json(path, lines=True)
    json_struct = json.loads(df.to_json(orient="records"))
    flat_df = pd.json_normalize(json_struct)
    if flat_df.empty:
        continue

    batch_id = path.name.split("_output.jsonl")[0]
    flat_df["model_response"] = flat_df["response.body.choices"].apply(
        lambda x: x[0]["message"]["content"] if isinstance(x, list) and x else None
    )
    flat_df["batch_id"] = batch_id
    flat_df["model"] = flat_df["response.body.model"]
    flat_df["fuzzy_model"] = flat_df["model"].apply(fuzzy_model)
    flat_df["prompt_tokens"] = flat_df["response.body.usage.prompt_tokens"]
    flat_df["completion_tokens"] = flat_df["response.body.usage.completion_tokens"]
    flat_df["total_tokens"] = flat_df["response.body.usage.total_tokens"]
    output_frames.append(flat_df)

outputs_df = pd.concat(output_frames, ignore_index=True)
outputs_df = outputs_df.drop(
    [col for col in outputs_df.columns if col.startswith("response")], axis=1
).drop(columns=["error"], errors="ignore")

input_frames = []
for f in inputs_glob:
    df = pd.read_json(f, lines=True)
    json_struct = json.loads(df.to_json(orient="records"))
    flat_df = pd.json_normalize(json_struct)
    if flat_df.empty:
        continue

    flat_df["input_file"] = f.name
    flat_df["fuzzy_model"] = flat_df["body.model"].apply(fuzzy_model)
    flat_df["metadata_key"] = flat_df["custom_id"].apply(metadata_key)
    flat_df["grammar_name"] = flat_df.get("body.metadata.grammar_name")
    flat_df["input_sentence"] = flat_df.get("body.metadata.input_sentence")
    flat_df["output_sentence"] = flat_df.get("body.metadata.output_sentence")
    flat_df["depth"] = flat_df.get("body.metadata.depth")
    flat_df["n_rules"] = flat_df.get("body.metadata.n_rules")
    flat_df["n_words"] = flat_df.get("body.metadata.n_words")
    input_frames.append(flat_df)

inputs_df = pd.concat(input_frames, ignore_index=True).drop(
    columns=["body.max_completion_tokens"], errors="ignore"
)

metadata_reference_df = (
    inputs_df.dropna(subset=["grammar_name", "input_sentence", "output_sentence"])
    .sort_values("input_file")
    .drop_duplicates(subset=["metadata_key"], keep="first")[
        [
            "metadata_key",
            "grammar_name",
            "input_sentence",
            "output_sentence",
            "depth",
            "n_rules",
            "n_words",
        ]
    ]
)

inputs_df = inputs_df.merge(
    metadata_reference_df,
    on="metadata_key",
    how="left",
    suffixes=("", "_reference"),
)
for column in [
    "grammar_name",
    "input_sentence",
    "output_sentence",
    "depth",
    "n_rules",
    "n_words",
]:
    ref_col = f"{column}_reference"
    inputs_df[column] = inputs_df[column].combine_first(inputs_df[ref_col])
    inputs_df = inputs_df.drop(columns=[ref_col])

io_df = pd.merge(
    outputs_df,
    inputs_df,
    on=["custom_id", "fuzzy_model"],
    how="inner",
)

io_df = io_df.drop([col for col in io_df.columns if col.startswith("body")], axis=1)

In [ ]:
io_df.info()

In [ ]:
grammar_blobs = (DATA_DIR / "size_exp").glob("grammar_*.json")
grammars = []

for path in grammar_blobs:
    with open(path, "r") as f:
        grammar = json.load(f)
        grammar_name = path.name.split("grammar_")[1].split(".json")[0]
        grammar["grammar_name"] = grammar_name
        grammar = pd.json_normalize(grammar)

        grammar["a.words"] = grammar[
            [
                "a.verbs",
                "a.nouns",
                "a.propns",
                "a.prons",
                "a.adjs",
                "a.det_def",
                "a.det_indef",
                "a.comps",
            ]
        ].apply(lambda row: sum(row, []), axis=1)
        grammar["b.words"] = grammar[
            [
                "b.verbs",
                "b.nouns",
                "b.propns",
                "b.prons",
                "b.adjs",
                "b.det_def",
                "b.det_indef",
                "b.comps",
            ]
        ].apply(lambda row: sum(row, []), axis=1)

        grammar = grammar.drop(
            columns=[
                "a.head_initial",
                "b.head_initial",
                "a.spec_initial",
                "b.spec_initial",
                "a.pro_drop",
                "b.pro_drop",
                "a.syllable_structure",
                "b.syllable_structure",
            ]
        )

        grammars.append(grammar)

grammars_df = pd.concat(grammars, ignore_index=True)

merged_df = pd.merge(
    io_df,
    grammars_df[["grammar_name", "a.words", "b.words"]],
    on="grammar_name",
)

merged_df["input_tokens"] = merged_df.apply(
    lambda row: len(
        tokenizers[fuzzy_model(row["model"])].encode(row["input_sentence"])
    ),
    axis=1,
)

merged_df["output_tokens"] = merged_df.apply(
    lambda row: len(
        tokenizers[fuzzy_model(row["model"])].encode(row["output_sentence"])
    ),
    axis=1,
)

### Extract Answer

In [ ]:
answer_re = re.compile(
    r"final\s*answer\s*(?::|-|—)?\s*(?:is\s*)?([^\n]+)",
    re.DOTALL | re.IGNORECASE,
)


def extract_answer(model_response):
    matches = answer_re.findall(model_response)
    if matches:
        last_match: str = matches[-1]
        # remove any punctuation characters,
        # but keep non-latin unicode chars
        last_match = re.sub(r"[^\w\s]", "", last_match, flags=re.UNICODE)
        last_match = last_match.strip()
        return last_match
    else:
        return None


merged_df = merged_df.drop_duplicates(subset=["custom_id", "batch_id"])
merged_df["model_answer"] = merged_df["model_response"].apply(extract_answer)
# treat empty strings as NaN
merged_df["model_answer"] = merged_df["model_answer"].replace("", np.nan)

In [ ]:
# Check how many entries there are where model_answer is null per model
print("Number of failed responses per model:")
null_answers = merged_df[merged_df["model_answer"].isna()]
null_answers.groupby(["fuzzy_model"]).size()

In [ ]:
# drop na
merged_df = merged_df[~merged_df["model_answer"].isna()]

merged_df.info()

### Compute Metrics

In [ ]:
# metrics
try:
    import sacrebleu
except ImportError:
    sacrebleu = None


def exact_match(row) -> bool:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return False
    return row["model_answer"] == row["output_sentence"]


def bow_match(row) -> bool:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return False
    return sorted(row["model_answer"].split()) == sorted(row["output_sentence"].split())


def edit_similarity(row) -> float:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return 1.0
    from strsimpy.jaro_winkler import JaroWinkler

    jw = JaroWinkler()
    return 1 - jw.distance(row["model_answer"], row["output_sentence"])


def bleu(row) -> float:
    if sacrebleu is None:
        return np.nan
    pred = row["model_answer"] or ""
    ref = row["output_sentence"] or ""
    return sacrebleu.sentence_bleu(pred, [ref]).score / 100.0


def num_oov_words(row) -> int:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return np.nan
    model_words: set[str] = set(row["model_answer"].split())
    output_words: set[str] = set(row["b.words"])
    oov_words: set[str] = model_words - output_words
    return len(oov_words)


# Metrics
merged_df["exact_match"] = merged_df.apply(exact_match, axis=1)
merged_df["bow_match"] = merged_df.apply(bow_match, axis=1)
merged_df["edit_similarity"] = merged_df.apply(edit_similarity, axis=1)
merged_df["num_oov_words"] = merged_df.apply(num_oov_words, axis=1)
merged_df["bleu"] = merged_df.apply(bleu, axis=1)

# manually calculate chrF, chrF++, and BLEU
preds = merged_df["model_answer"].tolist()
refs = merged_df["output_sentence"].tolist()

chrf = evaluate.load("chrf")
chrf_scores: list[float] = []
chrf_pp_scores: list[float] = []
for pred, ref in zip(preds, refs):
    if pred is None:
        pred = ""
    if ref is None:
        ref = ""
    chrf_scores.append(
        chrf.compute(
            predictions=[pred],
            references=[[ref]],
            beta=2,
            word_order=0,
        )
    )
    chrf_pp_scores.append(
        chrf.compute(
            predictions=[pred],
            references=[[ref]],
            beta=2,
            word_order=2,
        )
    )
merged_df["chrF"] = [score["score"] / 100.0 for score in chrf_scores]
merged_df["chrF++"] = [score["score"] / 100.0 for score in chrf_pp_scores]

merged_df["input_words"] = merged_df["input_sentence"].apply(lambda x: len(x.split()))
merged_df["input_words_rounded5"] = merged_df["input_words"].apply(
    lambda x: round(x / 5) * 5
)
merged_df["size"] = merged_df.apply(
    lambda row: int(row["n_rules"]) + int(row["n_words"]),
    axis=1,
)

merged_df["ttc"] = merged_df["completion_tokens"]
merged_df["ttc_binned"] = merged_df["ttc"].apply(
    # log2
    lambda x: round(np.log10(x)) if x > 0 else 0
)

metrics_df = merged_df.melt(
    id_vars=[
        "model",
        "fuzzy_model",
        "input_words_rounded5",
        "input_words",
        "custom_id",
        "model_answer",
        "output_sentence",
        "depth",
        "size",
        "ttc",
        "ttc_binned",
        "input_tokens",
        "output_tokens",
        "prompt_tokens",
    ],
    value_vars=[
        "exact_match",
        "bow_match",
        "edit_similarity",
        "num_oov_words",
        "bleu",
        "chrF",
        "chrF++",
    ],
    var_name="match_type",
    value_name="match_value",
)

# rename `exact_match` and `bow_match` in match_type column
metrics_df["match_type"] = metrics_df["match_type"].replace(
    {
        "exact_match": "Exact Match",
        "bow_match": "Bag of Words",
        "edit_similarity": "Edit Similarity",
        "num_oov_words": "Num OOV Words",
        "bleu": "BLEU Score",
        "chrF++": "chrF++",
        "chrF": "chrF",
    }
)

metrics_df["input_tokens_rounded"] = metrics_df["input_tokens"].apply(
    lambda x: round(x / 5) * 5
)
metrics_df["output_tokens_rounded"] = metrics_df["output_tokens"].apply(
    lambda x: round(x / 5) * 5
)
metrics_df["token_delta"] = abs(
    metrics_df["output_tokens"] - metrics_df["input_tokens"]
)
metrics_df["token_delta_norm"] = abs(
    metrics_df["output_tokens"] - metrics_df["input_tokens"]
) / metrics_df["input_tokens"].replace(0, np.nan)
metrics_df["token_delta_norm_rounded"] = metrics_df["token_delta_norm"].apply(
    lambda x: round(x, 1)
)

# convert the `model` column to an ordered categorical type
metrics_df["model_name"] = metrics_df["fuzzy_model"]
present_models = aes.ordered_models(metrics_df["model_name"].dropna().unique())
present_model_labels = [
    aes.MODEL_DISPLAY_NAMES.get(model, model) for model in present_models
]
model_palette = {
    aes.MODEL_DISPLAY_NAMES.get(model, model): aes.PALETTE_MODELS[model]
    for model in present_models
}
metrics_df["model_display_name"] = metrics_df["model_name"].map(
    lambda model: aes.MODEL_DISPLAY_NAMES.get(model, model)
)
metrics_df["model_display_name"] = pd.Categorical(
    metrics_df["model_display_name"],
    categories=present_model_labels,
    ordered=True,
)

In [ ]:
metrics_df["model_name"].unique()

## Answer Examination

In [ ]:
metrics_df[metrics_df["match_type"] == "Exact Match"].groupby(["match_type"])[
    "match_value"
].mean()

## Plots

### Grammatical & Sentential Complexity

In [ ]:
# Bin input_words into quintiles
N_BINS = 5
metrics_df["input_words_binned_quant"] = pd.qcut(
    metrics_df["input_words"],
    q=N_BINS,
    duplicates="drop",
)
metrics_df["input_words_binned_quant_num"] = metrics_df[
    "input_words_binned_quant"
].apply(lambda x: (x.left + x.right) / 2)

# round to nearest 5
metrics_df["input_words_binned"] = metrics_df["input_words"].apply(
    lambda x: round(x / 10) * 10
)

EXPERIMENT_NAME = "size"
selected_model = "all-models"
PERF_METRICS = ["Exact Match", "Bag of Words", "BLEU Score", "chrF++"]
PLOT_MODEL_ORDER = list(metrics_df["model_display_name"].cat.categories)
full_plot_source_df = metrics_df[metrics_df["match_type"].isin(PERF_METRICS)].copy()

if len(full_plot_source_df):
    fig = plt.figure(
        figsize=(aes.COLM_PAPER_WIDTH_IN, 1 * aes.FIG_HEIGHT_DOUBLEROW_DIFFAXES_IN)
    )
    grid = fig.add_gridspec(2, len(PERF_METRICS), wspace=0.12, hspace=0.7)

    for r, split in enumerate(["grammar", "sentence"]):
        for i, pm in enumerate(PERF_METRICS):
            ax = fig.add_subplot(grid[r, i])
            metric_df = full_plot_source_df[full_plot_source_df["match_type"] == pm]

            if split == "grammar":
                sns.lineplot(
                    data=metric_df,
                    x="size",
                    y="match_value",
                    hue="model_display_name",
                    style="model_display_name",
                    hue_order=PLOT_MODEL_ORDER,
                    style_order=PLOT_MODEL_ORDER,
                    markers=True,
                    ax=ax,
                    palette=model_palette,
                    legend=True,
                    errorbar="ci",
                )

                ax.set_ylim(-0.05, 1.05)
                ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
                ax.yaxis.grid(True, linestyle="--", alpha=0.5)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                ax.set_title(pm, fontweight="normal", loc="center")

                ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
                ax.set_xticks([0, 5000, 10_000])

                labels = ax.get_xticklabels()
                if labels:
                    labels[-1].set_ha("right")

                if i == 0:
                    ax.set_xlabel("Grammar size", loc="left")
                    ax.set_ylabel("Mean score")
                else:
                    ax.set_ylabel("")
                    ax.yaxis.set_ticklabels([])
                    ax.set_xlabel("")

                if i == len(PERF_METRICS) - 1:
                    ax.legend(
                        title="",
                        bbox_to_anchor=(1.05, 1.0),
                        loc="upper left",
                        borderaxespad=0,
                        frameon=False,
                    )
                else:
                    ax.get_legend().remove()
            else:
                sns.lineplot(
                    data=metric_df,
                    x="input_words_binned_quant_num",
                    y="match_value",
                    hue="model_display_name",
                    style="model_display_name",
                    hue_order=PLOT_MODEL_ORDER,
                    style_order=PLOT_MODEL_ORDER,
                    markers=True,
                    ax=ax,
                    palette=model_palette,
                    legend=True,
                    errorbar="ci",
                )

                ax.set_ylim(-0.05, 1.05)
                ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
                ax.yaxis.grid(True, linestyle="--", alpha=0.5)
                ax.spines["top"].set_visible(False)
                ax.spines["right"].set_visible(False)
                ax.set_title(None)

                ax.set_xticks([10, 20, 30, 40])

                if i == 0:
                    ax.set_xlabel("Sentence length (binned into quntiles)", loc="left")
                    ax.set_ylabel("Mean score")
                else:
                    ax.set_ylabel("")
                    ax.yaxis.set_ticklabels([])
                    ax.set_xlabel("")

                ax.get_legend().remove()

    plt.subplots_adjust(left=0, bottom=0, right=1, top=1)
    aes.save_figure(FIGURES_DIR / f"{EXPERIMENT_NAME}-full", fig=fig)
else:
    print("No full-metric data available for plotting.")

In [ ]:
exact_plot_source_df = metrics_df[metrics_df["match_type"] == "Exact Match"].copy()

if len(exact_plot_source_df):
    fig = plt.figure(figsize=(aes.COLM_PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN))
    grid = fig.add_gridspec(1, 2, wspace=0.1)

    ax = fig.add_subplot(grid[0, 0])
    sns.lineplot(
        data=exact_plot_source_df,
        x="size",
        y="match_value",
        hue="model_display_name",
        style="model_display_name",
        hue_order=PLOT_MODEL_ORDER,
        style_order=PLOT_MODEL_ORDER,
        markers=True,
        markersize=8,
        palette=model_palette,
        legend=True,
        errorbar="ci",
        ax=ax,
    )
    ax.set_title(None, fontweight="normal", loc="center")
    ax.set_xlabel("Grammar size")
    ax.set_ylabel("Exact match")
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
    ax.set_xticks([0, 5000, 10_000])
    labels = ax.get_xticklabels()
    if labels:
        labels[-1].set_ha("right")

    ax = fig.add_subplot(grid[0, 1])
    sns.lineplot(
        data=exact_plot_source_df,
        x="input_words_binned_quant_num",
        y="match_value",
        hue="model_display_name",
        style="model_display_name",
        hue_order=PLOT_MODEL_ORDER,
        style_order=PLOT_MODEL_ORDER,
        markers=True,
        markersize=8,
        palette=model_palette,
        legend=True,
        errorbar="ci",
        ax=ax,
    )
    ax.set_xlabel(f"Sentence length (binned into {N_BINS} quantiles)")
    ax.set_ylabel("")
    ax.set_ylim(-0.05, 1.05)
    ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
    ax.yaxis.grid(True, linestyle="--", alpha=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_ticklabels([])
    ax.set_xticks([10, 20, 30, 40])

    left_legend = fig.axes[0].get_legend()
    if left_legend is not None:
        left_legend.remove()

    right_legend = fig.axes[1].get_legend()
    if right_legend is not None:
        right_legend.set_title("")
        sns.move_legend(
            fig.axes[1], "upper left", bbox_to_anchor=(1.02, 1.0), frameon=False
        )

    plt.subplots_adjust(left=0, bottom=0, right=1, top=1)
    aes.save_figure(FIGURES_DIR / "size_overview_exact_match", fig=fig)
else:
    print("No exact-match data available for plotting.")

In [ ]:
EXPORT_DIR = NOTEBOOKS_DIR / "cache" / "combined-exp"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

plot_size_df = (
    exact_plot_source_df.copy() if len(exact_plot_source_df) else pd.DataFrame()
)
plot_length_df = (
    exact_plot_source_df.copy() if len(exact_plot_source_df) else pd.DataFrame()
)

if len(plot_size_df) and len(plot_length_df):
    export_size_df = plot_size_df.copy()
    export_size_df["experiment"] = EXPERIMENT_NAME
    export_size_df["panel"] = "grammar_size"
    export_size_df.to_csv(EXPORT_DIR / "size_grammar_size.csv", index=False)

    export_length_df = plot_length_df.copy()
    export_length_df["experiment"] = EXPERIMENT_NAME
    export_length_df["panel"] = "string_length"
    export_length_df.to_csv(EXPORT_DIR / "size_string_length.csv", index=False)

    print(f"Saved combined-exp inputs to {EXPORT_DIR}")
else:
    print("No size plot data available to export.")